# Chapter 1.6: Batching Strategies

## Learning Objectives

By the end of this notebook, you will:

1. **Understand** why batching is essential for LLM serving efficiency
2. **Implement** four batching strategies: no batching, static, dynamic, continuous
3. **Visualize** request timelines (Gantt charts) for each strategy
4. **Quantify** why continuous batching achieves 10-20x higher throughput
5. **Explain** iteration-level scheduling

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part1/chapter_1.6_batching.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part1/chapter_1.6_batching.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
from dataclasses import dataclass, field
from typing import List, Optional, Tuple
from collections import deque
import time

np.random.seed(42)
print("Ready to explore batching strategies!")

---

## 1. Why Batching Matters

Without batching (processing one request at a time):
- GPU reads all model weights from HBM for every single token
- For a 7B model: 14 GB read, but only ~8K FLOPs of useful work (one token)
- GPU compute utilization: < 0.1%

With batching (processing B requests simultaneously):
- GPU reads model weights once but processes B tokens
- Compute utilization increases by B times
- We amortize the memory bandwidth cost

The question is: **HOW** do we batch requests that arrive at different times and have different lengths?

In [ ]:
## Figure: Batching Strategies Comparison -- GPU Utilization
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

BLUE, GREEN, ORANGE, RED, PURPLE = '#4A90D9', '#27AE60', '#F39C12', '#E74C3C', '#8E44AD'
req_colors = ['#3498DB', '#E74C3C', '#2ECC71', '#F39C12', '#9B59B6', '#1ABC9C']

def draw_gantt(ax, title, requests, max_t=15, show_waste=False):
    """Draw a simplified Gantt chart.
    requests: list of (req_id, start, end, is_active_at_each_step)
    """
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Time Steps')
    ax.set_ylabel('GPU Slots')
    ax.set_xlim(0, max_t)
    ax.grid(True, alpha=0.2, axis='x')

# === Panel 1: No Batching ===
ax = axes[0, 0]
ax.set_title('No Batching (Sequential)', fontsize=13, fontweight='bold', color=RED)
ax.set_xlim(0, 18)
ax.set_ylim(-0.5, 1.5)
ax.set_yticks([0])
ax.set_yticklabels(['GPU'])

# Requests processed one at a time
starts = [0, 3, 5, 9, 11]
lengths = [3, 2, 4, 2, 3]
for i, (s, l) in enumerate(zip(starts, lengths)):
    ax.barh(0, l, left=s, height=0.6, color=req_colors[i], alpha=0.8, edgecolor='black')
    ax.text(s + l/2, 0, f'R{i}', ha='center', va='center', fontsize=9, fontweight='bold', color='white')
# Idle gaps
ax.barh(0, 14, left=0, height=0.6, color='lightgray', alpha=0.15, edgecolor='none')
ax.text(9, 1.0, 'GPU idle between requests', fontsize=9, color='gray', ha='center', style='italic')
ax.set_xlabel('Time')

# === Panel 2: Static Batching ===
ax = axes[0, 1]
ax.set_title('Static Batching (Wait for batch, pad to max)', fontsize=13, fontweight='bold', color=ORANGE)
ax.set_xlim(0, 14)
ax.set_ylim(-0.5, 4.5)
ax.set_yticks([0, 1, 2, 3])
ax.set_yticklabels([f'Slot {i}' for i in range(4)])

# Batch 1: 4 requests, max length = 5
batch1_lens = [3, 5, 2, 4]
for i, l in enumerate(batch1_lens):
    ax.barh(i, l, left=0, height=0.6, color=req_colors[i], alpha=0.8, edgecolor='black')
    ax.text(l/2, i, f'R{i}', ha='center', va='center', fontsize=9, fontweight='bold', color='white')
    # Padding waste
    if l < 5:
        ax.barh(i, 5-l, left=l, height=0.6, color='#E74C3C', alpha=0.2, edgecolor='red',
                linewidth=0.5, hatch='//')

# Batch 2: next 2 requests
batch2_lens = [4, 3]
for i, l in enumerate(batch2_lens):
    ax.barh(i, l, left=6, height=0.6, color=req_colors[i+4], alpha=0.8, edgecolor='black')
    ax.text(6+l/2, i, f'R{i+4}', ha='center', va='center', fontsize=9, fontweight='bold', color='white')

ax.axvline(x=5, color='red', linestyle='--', alpha=0.5)
ax.text(5, 4, 'Batch\nboundary', ha='center', fontsize=8, color='red')
ax.text(3.5, -0.4, 'Padding waste (GPU doing nothing)', fontsize=8, color=RED, style='italic')
ax.set_xlabel('Time')

# === Panel 3: Dynamic Batching ===
ax = axes[1, 0]
ax.set_title('Dynamic Batching (Flexible start, still padded)', fontsize=13, fontweight='bold', color=BLUE)
ax.set_xlim(0, 12)
ax.set_ylim(-0.5, 4.5)
ax.set_yticks([0, 1, 2, 3])
ax.set_yticklabels([f'Slot {i}' for i in range(4)])

# Batch 1: starts immediately with available requests
b1_lens = [3, 4, 2, 4]
for i, l in enumerate(b1_lens):
    ax.barh(i, l, left=0, height=0.6, color=req_colors[i], alpha=0.8, edgecolor='black')
    ax.text(l/2, i, f'R{i}', ha='center', va='center', fontsize=9, fontweight='bold', color='white')
    if l < 4:
        ax.barh(i, 4-l, left=l, height=0.6, color=RED, alpha=0.2, hatch='//')

# Batch 2: starts right after batch 1
b2_lens = [3, 2]
for i, l in enumerate(b2_lens):
    ax.barh(i, l, left=5, height=0.6, color=req_colors[i+4], alpha=0.8, edgecolor='black')
    ax.text(5+l/2, i, f'R{i+4}', ha='center', va='center', fontsize=9, fontweight='bold', color='white')

ax.text(6, -0.4, 'Less waiting, but still padding within batches', fontsize=8, color=BLUE, style='italic')
ax.set_xlabel('Time')

# === Panel 4: Continuous Batching ===
ax = axes[1, 1]
ax.set_title('Continuous Batching (No waste!)', fontsize=13, fontweight='bold', color=GREEN)
ax.set_xlim(0, 10)
ax.set_ylim(-0.5, 4.5)
ax.set_yticks([0, 1, 2, 3])
ax.set_yticklabels([f'Slot {i}' for i in range(4)])

# Continuous: when a request finishes, a new one takes its slot
schedule = [
    (0, 0, 3, 0),  # (slot, start, length, req_id)
    (1, 0, 4, 1),
    (2, 0, 2, 2),
    (3, 0, 5, 3),
    (2, 2, 3, 4),  # R4 takes slot 2 when R2 finishes!
    (0, 3, 2, 5),  # R5 takes slot 0 when R0 finishes!
    (2, 5, 3, 6),  # R6 takes slot 2
    (0, 5, 4, 7),  # R7 takes slot 0
]

for slot, start, length, rid in schedule:
    ax.barh(slot, length, left=start, height=0.6, color=req_colors[rid % len(req_colors)],
            alpha=0.8, edgecolor='black')
    ax.text(start + length/2, slot, f'R{rid}', ha='center', va='center',
            fontsize=8, fontweight='bold', color='white')

ax.text(5, -0.4, 'New requests fill slots immediately -- no padding, no waiting!',
        fontsize=8, color=GREEN, style='italic', fontweight='bold')
ax.set_xlabel('Time')

# Legend
fig.text(0.5, -0.02, 
         'Continuous batching achieves highest GPU utilization by eliminating padding waste and idle time.',
         ha='center', fontsize=12, fontweight='bold')

plt.suptitle('Batching Strategies: Impact on GPU Utilization', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nKey insight: Continuous batching is the breakthrough that enables high-throughput LLM serving.")
print("When a request finishes, its GPU slot is immediately reused -- no wasted cycles.")

In [ ]:
@dataclass
class SimRequest:
    """A request in our batching simulation."""
    id: int
    arrival_time: float
    prompt_tokens: int
    output_tokens: int
    
    # Filled during simulation
    start_time: float = -1
    first_token_time: float = -1
    end_time: float = -1
    
    # For visualization
    prefill_start: float = -1
    prefill_end: float = -1
    decode_intervals: list = field(default_factory=list)  # list of (start, end) for each decode step
    idle_intervals: list = field(default_factory=list)  # waiting periods (padding waste)


def generate_requests(n_requests=8, arrival_rate=2.0, seed=42):
    """Generate requests with Poisson arrivals and variable lengths."""
    np.random.seed(seed)
    requests = []
    t = 0
    for i in range(n_requests):
        t += np.random.exponential(1.0 / arrival_rate)
        prompt = np.random.choice([50, 100, 150, 200])
        output = np.random.choice([3, 5, 7, 10, 15])
        requests.append(SimRequest(id=i, arrival_time=t, prompt_tokens=prompt, output_tokens=output))
    return requests


# Constants for simulation
PREFILL_TIME_PER_TOKEN = 0.0001  # 0.1ms per prompt token (prefill is fast)
DECODE_TIME_BASE = 0.007         # 7ms base decode time
DECODE_TIME_PER_REQ = 0.0005     # 0.5ms additional per request in batch


requests = generate_requests(n_requests=8)
print("Generated requests:")
for r in requests:
    print(f"  Request {r.id}: arrival={r.arrival_time:.3f}s, "
          f"prompt={r.prompt_tokens} tokens, output={r.output_tokens} tokens")

---

## 2. Strategy 1: No Batching (Sequential)

Process one request at a time, start to finish, before moving to the next.

**Advantage**: Simple, lowest per-request latency (no sharing)

**Disadvantage**: Terrible throughput — GPU mostly idle reading weights for a single token

In [ ]:
def simulate_no_batching(requests: List[SimRequest]) -> List[SimRequest]:
    """Process requests one at a time."""
    results = []
    current_time = 0.0
    
    for req in sorted(requests, key=lambda r: r.arrival_time):
        r = SimRequest(**{k: v for k, v in req.__dict__.items() 
                         if k in ['id', 'arrival_time', 'prompt_tokens', 'output_tokens']})
        
        # Wait for request arrival
        current_time = max(current_time, r.arrival_time)
        r.start_time = current_time
        
        # Prefill
        prefill_time = r.prompt_tokens * PREFILL_TIME_PER_TOKEN
        r.prefill_start = current_time
        current_time += prefill_time
        r.prefill_end = current_time
        r.first_token_time = current_time
        
        # Decode (batch=1)
        for _ in range(r.output_tokens):
            step_time = DECODE_TIME_BASE + DECODE_TIME_PER_REQ  # batch=1
            start = current_time
            current_time += step_time
            r.decode_intervals.append((start, current_time))
        
        r.end_time = current_time
        results.append(r)
    
    return results


no_batch_results = simulate_no_batching(requests)
total_time = max(r.end_time for r in no_batch_results) - min(r.arrival_time for r in no_batch_results)
total_tokens = sum(r.output_tokens for r in no_batch_results)
print(f"No Batching: {total_tokens} tokens in {total_time:.3f}s = {total_tokens/total_time:.0f} tok/s")

---

## 3. Strategy 2: Static Batching

Collect requests until we have `batch_size` requests, then process them all together. Wait for ALL to finish before starting the next batch.

**Problem**: Shorter requests must wait for the longest request in the batch. This wastes GPU time on padding.

In [ ]:
def simulate_static_batching(requests: List[SimRequest], batch_size: int = 4) -> List[SimRequest]:
    """Process requests in fixed-size batches. All must finish before next batch."""
    results = []
    sorted_reqs = sorted(requests, key=lambda r: r.arrival_time)
    current_time = 0.0
    
    i = 0
    while i < len(sorted_reqs):
        # Collect a batch
        batch = []
        while len(batch) < batch_size and i < len(sorted_reqs):
            batch.append(i)
            i += 1
        
        # Wait for all requests in batch to arrive
        batch_reqs = []
        for idx in batch:
            r = SimRequest(**{k: v for k, v in sorted_reqs[idx].__dict__.items()
                             if k in ['id', 'arrival_time', 'prompt_tokens', 'output_tokens']})
            current_time = max(current_time, r.arrival_time)
            batch_reqs.append(r)
        
        # Prefill all requests in batch
        for r in batch_reqs:
            r.start_time = current_time
            prefill_time = r.prompt_tokens * PREFILL_TIME_PER_TOKEN
            r.prefill_start = current_time
            current_time += prefill_time
            r.prefill_end = current_time
            r.first_token_time = current_time
        
        # Decode: run until the LONGEST request in batch finishes
        max_output = max(r.output_tokens for r in batch_reqs)
        bs = len(batch_reqs)
        
        for step in range(max_output):
            step_time = DECODE_TIME_BASE + bs * DECODE_TIME_PER_REQ
            start = current_time
            current_time += step_time
            
            for r in batch_reqs:
                if step < r.output_tokens:
                    r.decode_intervals.append((start, current_time))
                else:
                    # This request is done but still occupying a slot (waste!)
                    r.idle_intervals.append((start, current_time))
        
        for r in batch_reqs:
            r.end_time = r.decode_intervals[-1][1] if r.decode_intervals else current_time
            results.append(r)
    
    return results


static_results = simulate_static_batching(requests, batch_size=4)
total_time = max(r.end_time for r in static_results) - min(r.arrival_time for r in static_results)
total_tokens = sum(r.output_tokens for r in static_results)
print(f"Static Batching (bs=4): {total_tokens} tokens in {total_time:.3f}s = {total_tokens/total_time:.0f} tok/s")

---

## 4. Strategy 3: Dynamic Batching

Like static batching, but don't wait for a full batch — start processing when either:
1. We have `batch_size` requests, OR
2. A timeout expires

Still waits for the longest request in the batch before starting a new batch.

In [ ]:
def simulate_dynamic_batching(
    requests: List[SimRequest], 
    max_batch_size: int = 4,
    max_wait_time: float = 0.1  # max seconds to wait for more requests
) -> List[SimRequest]:
    """Dynamic batching: start when batch is full OR timeout."""
    results = []
    sorted_reqs = list(sorted(requests, key=lambda r: r.arrival_time))
    current_time = 0.0
    req_idx = 0
    
    while req_idx < len(sorted_reqs):
        # Wait for at least one request
        current_time = max(current_time, sorted_reqs[req_idx].arrival_time)
        
        # Collect requests (up to batch_size or timeout)
        deadline = current_time + max_wait_time
        batch_reqs = []
        
        while (req_idx < len(sorted_reqs) and 
               len(batch_reqs) < max_batch_size and 
               sorted_reqs[req_idx].arrival_time <= deadline):
            r = SimRequest(**{k: v for k, v in sorted_reqs[req_idx].__dict__.items()
                             if k in ['id', 'arrival_time', 'prompt_tokens', 'output_tokens']})
            batch_reqs.append(r)
            req_idx += 1
        
        if not batch_reqs:
            continue
        
        # Prefill
        for r in batch_reqs:
            r.start_time = current_time
            prefill_time = r.prompt_tokens * PREFILL_TIME_PER_TOKEN
            r.prefill_start = current_time
            current_time += prefill_time
            r.prefill_end = current_time
            r.first_token_time = current_time
        
        # Decode
        max_output = max(r.output_tokens for r in batch_reqs)
        bs = len(batch_reqs)
        
        for step in range(max_output):
            step_time = DECODE_TIME_BASE + bs * DECODE_TIME_PER_REQ
            start = current_time
            current_time += step_time
            
            for r in batch_reqs:
                if step < r.output_tokens:
                    r.decode_intervals.append((start, current_time))
                else:
                    r.idle_intervals.append((start, current_time))
        
        for r in batch_reqs:
            r.end_time = r.decode_intervals[-1][1]
            results.append(r)
    
    return results


dynamic_results = simulate_dynamic_batching(requests, max_batch_size=4, max_wait_time=0.1)
total_time = max(r.end_time for r in dynamic_results) - min(r.arrival_time for r in dynamic_results)
total_tokens = sum(r.output_tokens for r in dynamic_results)
print(f"Dynamic Batching (bs=4): {total_tokens} tokens in {total_time:.3f}s = {total_tokens/total_time:.0f} tok/s")

---

## 5. Strategy 4: Continuous Batching (In-Flight Batching)

### The Breakthrough

Continuous batching (proposed in the Orca paper) makes a simple but powerful change:

**Instead of waiting for the entire batch to finish, insert new requests into the batch at every decode step.**

When a request finishes:
1. Its slot is immediately freed
2. A waiting request can take its slot at the next iteration
3. No GPU cycles are wasted on padding

This is also called **iteration-level scheduling** because scheduling decisions happen at every decode iteration.

In [ ]:
def simulate_continuous_batching(
    requests: List[SimRequest],
    max_batch_size: int = 4
) -> List[SimRequest]:
    """Continuous batching: insert/remove requests at every decode step."""
    results = []
    waiting = deque(sorted(requests, key=lambda r: r.arrival_time))
    active = []  # requests currently decoding
    current_time = 0.0
    
    # Create fresh copies
    waiting_fresh = deque()
    for r in waiting:
        waiting_fresh.append(SimRequest(
            id=r.id, arrival_time=r.arrival_time,
            prompt_tokens=r.prompt_tokens, output_tokens=r.output_tokens
        ))
    waiting = waiting_fresh
    
    # Track remaining tokens for active requests
    remaining = {}  # request id -> remaining output tokens
    
    while waiting or active:
        # Add new requests that have arrived and we have capacity
        while (waiting and 
               len(active) < max_batch_size and 
               waiting[0].arrival_time <= current_time):
            r = waiting.popleft()
            
            # Prefill this request
            r.start_time = current_time
            prefill_time = r.prompt_tokens * PREFILL_TIME_PER_TOKEN
            r.prefill_start = current_time
            current_time += prefill_time
            r.prefill_end = current_time
            r.first_token_time = current_time
            
            active.append(r)
            remaining[r.id] = r.output_tokens
        
        if not active:
            if waiting:
                current_time = waiting[0].arrival_time
            continue
        
        # Decode step for all active requests
        bs = len(active)
        step_time = DECODE_TIME_BASE + bs * DECODE_TIME_PER_REQ
        start = current_time
        current_time += step_time
        
        # Update all active requests
        finished = []
        for r in active:
            r.decode_intervals.append((start, current_time))
            remaining[r.id] -= 1
            if remaining[r.id] <= 0:
                r.end_time = current_time
                finished.append(r)
        
        # Remove finished requests — their slots are now FREE for new requests!
        for r in finished:
            active.remove(r)
            del remaining[r.id]
            results.append(r)
    
    return results


continuous_results = simulate_continuous_batching(requests, max_batch_size=4)
total_time = max(r.end_time for r in continuous_results) - min(r.arrival_time for r in continuous_results)
total_tokens = sum(r.output_tokens for r in continuous_results)
print(f"Continuous Batching (bs=4): {total_tokens} tokens in {total_time:.3f}s = {total_tokens/total_time:.0f} tok/s")

---

## 6. Visualizing Request Timelines (Gantt Charts)

In [ ]:
def plot_gantt_chart(results: List[SimRequest], title: str, ax=None):
    """Plot a Gantt chart showing request processing timeline."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(16, max(4, len(results) * 0.6)))
    
    # Sort by ID for consistent display
    results = sorted(results, key=lambda r: r.id)
    
    colors = plt.cm.Set3(np.linspace(0, 1, len(results)))
    
    for i, r in enumerate(results):
        y = i
        height = 0.7
        
        # Arrival marker
        ax.plot(r.arrival_time, y, 'k>', markersize=8)
        
        # Queue time (waiting)
        if r.start_time > r.arrival_time:
            ax.barh(y, r.start_time - r.arrival_time, left=r.arrival_time,
                    height=height, color='#bdc3c7', alpha=0.5, edgecolor='gray', linewidth=0.5)
        
        # Prefill
        if r.prefill_end > r.prefill_start:
            ax.barh(y, r.prefill_end - r.prefill_start, left=r.prefill_start,
                    height=height, color='#3498db', alpha=0.8, edgecolor='black', linewidth=0.5)
        
        # Decode steps
        for start, end in r.decode_intervals:
            ax.barh(y, end - start, left=start,
                    height=height, color=colors[i], alpha=0.8, edgecolor='black', linewidth=0.5)
        
        # Idle/padding (waste in static batching)
        for start, end in r.idle_intervals:
            ax.barh(y, end - start, left=start,
                    height=height, color='#e74c3c', alpha=0.3, edgecolor='red', 
                    linewidth=0.5, hatch='//')
    
    # Legend
    legend_elements = [
        mpatches.Patch(facecolor='#bdc3c7', alpha=0.5, label='Queue wait'),
        mpatches.Patch(facecolor='#3498db', alpha=0.8, label='Prefill'),
        mpatches.Patch(facecolor='#2ecc71', alpha=0.8, label='Decode'),
        mpatches.Patch(facecolor='#e74c3c', alpha=0.3, label='Padding (waste)'),
    ]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=9)
    
    ax.set_yticks(range(len(results)))
    ax.set_yticklabels([f'Req {r.id} ({r.output_tokens} tok)' for r in results], fontsize=10)
    ax.set_xlabel('Time (seconds)', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    ax.invert_yaxis()
    
    return ax


# Plot all four strategies
fig, axes = plt.subplots(4, 1, figsize=(18, 20))

plot_gantt_chart(no_batch_results, 'Strategy 1: No Batching (Sequential)', ax=axes[0])
plot_gantt_chart(static_results, 'Strategy 2: Static Batching (batch=4)', ax=axes[1])
plot_gantt_chart(dynamic_results, 'Strategy 3: Dynamic Batching (batch=4)', ax=axes[2])
plot_gantt_chart(continuous_results, 'Strategy 4: Continuous Batching (batch=4)', ax=axes[3])

plt.tight_layout()
plt.show()

In [ ]:
# Comprehensive comparison

def compute_metrics(results: List[SimRequest]) -> dict:
    total_time = max(r.end_time for r in results) - min(r.arrival_time for r in results)
    total_output_tokens = sum(r.output_tokens for r in results)
    
    ttfts = [(r.first_token_time - r.arrival_time) * 1000 for r in results]
    e2e = [(r.end_time - r.arrival_time) * 1000 for r in results]
    
    # Compute wasted time (idle/padding)
    total_idle = sum(sum(end - start for start, end in r.idle_intervals) for r in results)
    total_active = sum(sum(end - start for start, end in r.decode_intervals) for r in results)
    
    return {
        'throughput': total_output_tokens / total_time,
        'ttft_mean': np.mean(ttfts),
        'ttft_p99': np.percentile(ttfts, 99),
        'e2e_mean': np.mean(e2e),
        'e2e_p99': np.percentile(e2e, 99),
        'total_time': total_time,
        'gpu_utilization': total_active / (total_active + total_idle) * 100 if (total_active + total_idle) > 0 else 100,
        'waste_ratio': total_idle / (total_active + total_idle) * 100 if (total_active + total_idle) > 0 else 0,
    }


all_results = {
    'No Batching': no_batch_results,
    'Static (bs=4)': static_results,
    'Dynamic (bs=4)': dynamic_results,
    'Continuous (bs=4)': continuous_results,
}

print("Comprehensive Comparison")
print("=" * 90)
print(f"{'Strategy':<22} {'Throughput':>12} {'TTFT Mean':>12} {'E2E Mean':>12} {'Total Time':>12} {'Waste':>10}")
print("-" * 90)

metrics_all = {}
for name, res in all_results.items():
    m = compute_metrics(res)
    metrics_all[name] = m
    print(f"{name:<22} {m['throughput']:>9.0f} t/s {m['ttft_mean']:>9.0f} ms {m['e2e_mean']:>9.0f} ms "
          f"{m['total_time']:>9.3f} s {m['waste_ratio']:>7.1f}%")

In [ ]:
# Bar chart comparison

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

strategies = list(metrics_all.keys())
colors = ['#e74c3c', '#f39c12', '#3498db', '#2ecc71']

# Throughput
throughputs = [metrics_all[s]['throughput'] for s in strategies]
bars = axes[0].bar(strategies, throughputs, color=colors, alpha=0.8)
axes[0].set_ylabel('Throughput (tokens/s)', fontsize=12)
axes[0].set_title('Throughput Comparison', fontsize=14)
for bar, val in zip(bars, throughputs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                 f'{val:.0f}', ha='center', va='bottom', fontweight='bold')
axes[0].tick_params(axis='x', rotation=20)

# TTFT
ttfts = [metrics_all[s]['ttft_mean'] for s in strategies]
bars = axes[1].bar(strategies, ttfts, color=colors, alpha=0.8)
axes[1].set_ylabel('TTFT Mean (ms)', fontsize=12)
axes[1].set_title('Time to First Token', fontsize=14)
for bar, val in zip(bars, ttfts):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                 f'{val:.0f}', ha='center', va='bottom', fontweight='bold')
axes[1].tick_params(axis='x', rotation=20)

# Waste ratio
wastes = [metrics_all[s]['waste_ratio'] for s in strategies]
bars = axes[2].bar(strategies, wastes, color=colors, alpha=0.8)
axes[2].set_ylabel('Wasted GPU Time (%)', fontsize=12)
axes[2].set_title('GPU Waste (Padding)', fontsize=14)
for bar, val in zip(bars, wastes):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                 f'{val:.1f}%', ha='center', va='bottom', fontweight='bold')
axes[2].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

---

## 7. Iteration-Level Scheduling Explained

### Why Continuous Batching Wins

```
Static Batching:                    Continuous Batching:
                                    
Batch 1:  [R1 R2 R3 R4]           Iter 1: [R1 R2 R3 R4]    decode
          [R1 R2 R3 R4]           Iter 2: [R1 R2 R3 R4]    decode
          [R1 R2 __ R4]  padding!  Iter 3: [R1 R2 R5 R4]    R3 done, R5 joins!
          [R1 __ __ R4]  padding!  Iter 4: [R1 R6 R5 R4]    R2 done, R6 joins!
          [R1 __ __ __]  padding!  Iter 5: [R1 R6 R5 __]    R4 done
--- wait for batch 1 to finish --- Iter 6: [R1 R6 R5 R7]    R7 joins!
Batch 2:  [R5 R6 R7 R8]           ...no waiting, continuous flow
```

**The key insight**: In static batching, when request R3 finishes early, its GPU slot is wasted until all requests in the batch finish. In continuous batching, R5 immediately takes R3's slot.

In [ ]:
# Large-scale comparison with more requests

n_requests_large = 100

large_results = {}

for bs in [4, 8, 16, 32]:
    requests_large = generate_requests(n_requests=n_requests_large, arrival_rate=5.0, seed=42)
    
    # Static
    static = simulate_static_batching(requests_large, batch_size=bs)
    static_m = compute_metrics(static)
    
    # Continuous
    requests_large = generate_requests(n_requests=n_requests_large, arrival_rate=5.0, seed=42)
    continuous = simulate_continuous_batching(requests_large, max_batch_size=bs)
    continuous_m = compute_metrics(continuous)
    
    large_results[bs] = {
        'static': static_m,
        'continuous': continuous_m
    }
    
    speedup = continuous_m['throughput'] / static_m['throughput']
    print(f"Batch={bs}: Static={static_m['throughput']:.0f} tok/s, "
          f"Continuous={continuous_m['throughput']:.0f} tok/s, "
          f"Speedup={speedup:.1f}x")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bs_list = list(large_results.keys())
static_tp = [large_results[bs]['static']['throughput'] for bs in bs_list]
continuous_tp = [large_results[bs]['continuous']['throughput'] for bs in bs_list]

x = np.arange(len(bs_list))
width = 0.35

axes[0].bar(x - width/2, static_tp, width, label='Static Batching', color='#e74c3c', alpha=0.8)
axes[0].bar(x + width/2, continuous_tp, width, label='Continuous Batching', color='#2ecc71', alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels([f'bs={bs}' for bs in bs_list])
axes[0].set_ylabel('Throughput (tokens/s)', fontsize=12)
axes[0].set_title('Throughput: Static vs Continuous', fontsize=14)
axes[0].legend()

speedups = [c/s for s, c in zip(static_tp, continuous_tp)]
axes[1].bar(bs_list, speedups, color='#9b59b6', alpha=0.8)
axes[1].set_xlabel('Max Batch Size')
axes[1].set_ylabel('Speedup (x)', fontsize=12)
axes[1].set_title('Continuous Batching Speedup over Static', fontsize=14)
for b, s in zip(bs_list, speedups):
    axes[1].text(b, s + 0.05, f'{s:.1f}x', ha='center', fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 8. Practical Considerations

### 8.1 Prefill-Decode Interference

In continuous batching, prefilling a new request while other requests are decoding can cause a latency spike for the decoding requests (because prefill is compute-heavy).

**Solutions**:
- **Chunked prefill**: Split long prefills into smaller chunks
- **Disaggregated serving**: Separate prefill and decode to different GPUs
- **Scheduling policies**: Limit how often prefill interrupts decode

### 8.2 KV-Cache Memory Management

Continuous batching requires dynamic KV-cache allocation since requests join and leave the batch at different times. This is exactly what **PagedAttention** (vLLM) solves.

In [ ]:
# Visualize batch composition over time (continuous batching)

def plot_batch_composition(results: List[SimRequest], title='Batch Composition Over Time'):
    """Show which requests are in the batch at each time step."""
    # Collect all time points
    time_points = set()
    for r in results:
        for start, end in r.decode_intervals:
            time_points.add(start)
            time_points.add(end)
    time_points = sorted(time_points)
    
    # For each time point, find which requests are active
    batch_sizes = []
    active_reqs = []
    
    for t in time_points:
        active = []
        for r in results:
            for start, end in r.decode_intervals:
                if start <= t < end:
                    active.append(r.id)
                    break
        batch_sizes.append(len(active))
        active_reqs.append(active)
    
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.step(time_points, batch_sizes, 'b-', linewidth=2, where='post')
    ax.fill_between(time_points, batch_sizes, alpha=0.3, step='post', color='blue')
    ax.set_xlabel('Time (s)', fontsize=12)
    ax.set_ylabel('Active Batch Size', fontsize=12)
    ax.set_title(title, fontsize=14)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, max(batch_sizes) + 1)
    plt.tight_layout()
    plt.show()


plot_batch_composition(continuous_results, 'Continuous Batching: Active Batch Size Over Time')

---

## Exercises

### Exercise 1: Implement a Continuous Batching Simulator

Extend the continuous batching simulator with:
1. Chunked prefill (split long prefills into chunks of `chunk_size` tokens)
2. Priority scheduling (prioritize shorter requests)

In [ ]:
def simulate_continuous_batching_v2(
    requests: List[SimRequest],
    max_batch_size: int = 8,
    prefill_chunk_size: int = 128,  # max tokens to prefill at once
    priority: str = 'fifo'  # 'fifo' or 'shortest_first'
) -> List[SimRequest]:
    """
    Exercise: Advanced continuous batching with:
    - Chunked prefill
    - Priority scheduling
    
    For chunked prefill:
    - If a prompt has 500 tokens and chunk_size=128,
      prefill in 4 chunks (128, 128, 128, 116)
    - Between chunks, run one decode step for active requests
    - This reduces the latency spike for decoding requests
    """
    # YOUR CODE HERE
    pass

# Test:
# results_v2 = simulate_continuous_batching_v2(requests, max_batch_size=4)
# m = compute_metrics(results_v2)
# print(f"Throughput: {m['throughput']:.0f} tok/s")

### Exercise 2: Compare Strategies Under Different Workloads

Create workloads with:
1. Uniform output lengths (all requests generate same number of tokens)
2. Highly variable output lengths (some requests generate 5 tokens, others 500)

Compare how the throughput advantage of continuous batching changes.

In [ ]:
# Exercise 2: Workload comparison

# YOUR CODE HERE
# Hint: Continuous batching wins MORE when output lengths are highly variable
# because static batching wastes more time on padding.

---

## Summary

### Key Takeaways

1. **No batching** has the worst throughput — GPU reads all weights per single token.

2. **Static batching** improves throughput but wastes GPU time on padding when requests have different output lengths.

3. **Dynamic batching** reduces waiting time but still suffers from padding waste.

4. **Continuous batching** eliminates padding waste by inserting/removing requests at every decode iteration. This can achieve 10-20x higher throughput than static batching.

5. **Iteration-level scheduling** is the key innovation: scheduling decisions happen at every decode step, not at batch boundaries.

6. **Trade-offs**: Continuous batching requires dynamic KV-cache management (PagedAttention) and careful handling of prefill-decode interference.

### What's Next

In **Chapter 1.7: Quantization Primer**, we'll learn how to reduce model size and increase throughput by using lower-precision number formats.